# Практика 2. Проверка оплат и первая покупка клиента

Запустите Compose из этой папки и прочитайте [план](Практика_2_план.md). Ячейки выполняйте сверху вниз. Параметры соединения указаны отдельно в следующей ячейке; все `todo` нужно дополнить.

In [1]:
# Установите драйвер PostgreSQL для Python в текущем ядре ноутбука.
%pip install -q "psycopg[binary]" pandas

# Библиотека psycopg открывает соединение и выполняет SQL-запросы.
import psycopg

# pandas собирает полученные строки в таблицу для просмотра.
import pandas as pd

# Параметры подключения можно изменить здесь, например если занят порт 5435.
pg_host = "localhost"
pg_port = 5435
pg_database = "shop"
pg_user = "student"
pg_password = "studentpass"


Note: you may need to restart the kernel to use updated packages.


## Проверка подключения к PostgreSQL

Выполните эту ячейку до заданий. Результат должен содержать имя базы `shop` и число `1`.

In [2]:
# connect открывает соединение с параметрами из переменных выше.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект для выполнения SQL.
    with conn.cursor() as cur:
        # execute отправляет простой проверочный запрос.
        cur.execute("select current_database(), 1;")
        # fetchone получает единственную строку ответа.
        print(cur.fetchone())


('shop', 1)


## 1. Проверка источников

Посчитайте строки в `customers`, `orders`, `payments` и заказы по статусам. Сравните результаты с таблицей самопроверки в плане. Первые два запроса показывают простой вывод кортежей; начиная с третьего, результат получает имена столбцов и отображается как таблица pandas.

In [3]:
sql = """
select count(*) as customers_count from customers;
"""

# connect открывает соединение; cursor отправляет SQL в PostgreSQL.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        # Курсор возвращает строки результата по одной.
        for row in cur:
            print(row)


UndefinedTable: relation "customers" does not exist
LINE 2: select count(*) as customers_count from customers;
                                                ^

In [5]:
sql = """
select count(*) as orders_count from orders;
"""

# connect открывает соединение; cursor отправляет SQL в PostgreSQL.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
        # Курсор возвращает строки результата по одной.
        for row in cur:
            print(row)


(141,)


In [ ]:
# SELECT считает строки в payments; результат прочитаем как таблицу.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select count(*) as payments_count from payments;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


In [ ]:
# GROUP BY считает заказы для каждого статуса; ORDER BY сортирует группы.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select o.status, count(*) as orders_count
from orders as o
group by o.status
order by o.status;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


## 2. Почему платежи размножают заказы

Сравните `total_amount` заказов 113, 116 и 119 со строками платежей.

In [ ]:
# LEFT JOIN сохраняет заказ без платежа и показывает отдельную строку для каждой операции.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select o.order_id, o.status, o.total_amount,
       p.payment_id, p.payment_status, p.amount
from orders as o
left join payments as p on p.order_id = o.order_id
where o.order_id in (113, 116, 119)
order by o.order_id, p.payment_id;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


## 3. Витрина контроля оплат

Одна строка на заказ со статусом `paid`. В `create table` заполнен только первичный ключ: допишите остальные поля по плану и удалите комментарий `todo`. Затем отдельно очистите витрину и выполните вставку.

In [ ]:
# CREATE TABLE создаёт витрину; первичный ключ не допускает повторения order_id.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
create table if not exists practice_paid_order_checks (
    order_id integer primary key
    -- todo: добавьте customer_id, total_amount, first_paid_at,
    -- successful_operations_count, successful_amount
);
'''

# Не запускаем шаблон, пока в SQL остаётся незаполненное задание.
if "-- todo" in sql:
    raise NotImplementedError("Допишите поля таблицы перед созданием")

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)

# Если ошибок не было, команда выполнена и изменение сохранено в базе.
print("Команда выполнена")


In [ ]:
# TRUNCATE очищает витрину перед повторной загрузкой.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
truncate table practice_paid_order_checks;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)

# Если ошибок не было, команда выполнена и изменение сохранено в базе.
print("Команда выполнена")


In [ ]:
# CTE агрегирует платежи до одной строки на заказ; INSERT загружает paid-заказы в витрину.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
with successful_payments as (
    select p.order_id,
           -- todo: замените null на время первой успешной операции
           null::timestamp as first_paid_at,
           count(*) as successful_operations_count,
           sum(p.amount) as successful_amount
    from payments as p
    where p.payment_status = 'success'
    group by p.order_id
)
insert into practice_paid_order_checks (
    order_id, customer_id, total_amount, first_paid_at,
    successful_operations_count, successful_amount
)
select o.order_id, o.customer_id, o.total_amount,
       sp.first_paid_at,
       coalesce(sp.successful_operations_count, 0),
       coalesce(sp.successful_amount, 0)
from orders as o
left join successful_payments as sp on sp.order_id = o.order_id
where o.status = 'paid';
'''

# Не запускаем шаблон, пока в SQL остаётся незаполненное задание.
if "-- todo" in sql:
    raise NotImplementedError("Допишите время первой успешной операции")

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)

# Если ошибок не было, команда выполнена и изменение сохранено в базе.
print("Команда выполнена")


In [ ]:
# FILTER считает отдельно подтверждённые заказы, а SUM складывает их суммы.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select count(*) as paid_orders,
       count(*) filter (where successful_operations_count > 0) as confirmed_orders,
       count(*) filter (where successful_operations_count = 0) as paid_without_success,
       sum(total_amount) filter (where successful_operations_count > 0) as confirmed_orders_amount
from practice_paid_order_checks;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


In [ ]:
# Проверяем повторный платёж, оплату частями и заказ без успешной операции.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select * from practice_paid_order_checks
where order_id in (113, 116, 119)
order by order_id;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


## 4. История подтверждённых покупок

Допишите `row_number` и `lag`. Оконные `count` и `sum` уже даны как пример. Результат остаётся на уровне отдельных заказов.

In [ ]:
# Оконные функции дополняют каждую покупку номером, предыдущей датой и итогами клиента.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select pc.customer_id, pc.order_id, pc.first_paid_at,
       -- todo: замените null на row_number по клиенту
       null::bigint as purchase_number,
       -- todo: замените null на lag времени оплаты
       null::timestamp as previous_paid_at,
       count(*) over (partition by pc.customer_id) as paid_orders_count,
       sum(pc.total_amount) over (partition by pc.customer_id) as paid_orders_amount
from practice_paid_order_checks as pc
where pc.successful_operations_count > 0
  and pc.customer_id in (1, 2)
order by pc.customer_id, pc.first_paid_at, pc.order_id;
'''

# Не запускаем шаблон, пока в SQL остаётся незаполненное задание.
if "-- todo" in sql:
    raise NotImplementedError("Допишите row_number и lag для истории покупок")

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


## 5. Витрина первой покупки клиента

Одна строка на клиента. Допишите поля в `create table`, затем очистите витрину. Во вставке нужен один CTE с `row_number`; условие первой покупки остаётся в `on` соединения.

In [ ]:
# CREATE TABLE создаёт клиентскую витрину с одной строкой на customer_id.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
create table if not exists practice_customer_first_purchase (
    customer_id integer primary key
    -- todo: добавьте city, first_paid_order_id,
    -- first_paid_at, hours_to_first_paid
);
'''

# Не запускаем шаблон, пока в SQL остаётся незаполненное задание.
if "-- todo" in sql:
    raise NotImplementedError("Допишите поля таблицы перед созданием")

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)

# Если ошибок не было, команда выполнена и изменение сохранено в базе.
print("Команда выполнена")


In [ ]:
# TRUNCATE очищает клиентскую витрину перед повторной загрузкой.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
truncate table practice_customer_first_purchase;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)

# Если ошибок не было, команда выполнена и изменение сохранено в базе.
print("Команда выполнена")


In [ ]:
# CTE нумерует покупки; LEFT JOIN и условие в ON сохраняют клиентов без покупки.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
with numbered_orders as (
    select pc.customer_id, pc.order_id, pc.first_paid_at,
           -- todo: замените null на row_number по клиенту
           null::bigint as purchase_number
    from practice_paid_order_checks as pc
    where pc.successful_operations_count > 0
)
insert into practice_customer_first_purchase (
    customer_id, city, first_paid_order_id, first_paid_at,
    hours_to_first_paid
)
select c.customer_id, c.city, no.order_id, no.first_paid_at,
       -- todo: часы от регистрации до первой успешной оплаты
       null::numeric
from customers as c
left join numbered_orders as no
    on no.customer_id = c.customer_id
   and no.purchase_number = 1;
'''

# Не запускаем шаблон, пока в SQL остаётся незаполненное задание.
if "-- todo" in sql:
    raise NotImplementedError("Допишите номер покупки и интервал до первой оплаты")

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)

# Если ошибок не было, команда выполнена и изменение сохранено в базе.
print("Команда выполнена")


## 6. Самопроверка

Сверьте обе витрины с ожидаемыми результатами в плане. Повторите очистку и загрузку обеих витрин: число строк не должно измениться.

In [ ]:
# COUNT и FILTER сверяют число клиентов с покупкой и без неё.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select count(*) as mart_rows,
       count(*) filter (where first_paid_order_id is not null) as activated_customers,
       count(*) filter (where first_paid_order_id is null) as customers_without_purchase
from practice_customer_first_purchase;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


In [ ]:
# Просматриваем контрольные строки для двух покупателей и клиента без покупки.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select * from practice_customer_first_purchase
where customer_id in (1, 2, 40)
order by customer_id;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


In [ ]:
# GROUP BY и HAVING ищут клиентов с несколькими строками; ответ должен быть пустым.
# Строка sql хранит запрос, который будет отправлен в PostgreSQL.
sql = '''
select customer_id, count(*) as rows_per_customer
from practice_customer_first_purchase
group by customer_id
having count(*) > 1;
'''

# connect открывает соединение; при выходе из with закрывает его и фиксирует успешные изменения.
with psycopg.connect(
    host=pg_host,
    port=pg_port,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
) as conn:
    # cursor создаёт объект, через который мы отправляем SQL и читаем ответ.
    with conn.cursor() as cur:
        # execute выполняет SQL, записанный в переменной sql.
        cur.execute(sql)
        # description содержит имена столбцов результата SELECT.
        columns = [column.name for column in cur.description]
        # fetchall загружает все строки результата в список кортежей.
        rows = cur.fetchall()
        # zip связывает имена столбцов со значениями; dict делает строку словарём.
        records = [dict(zip(columns, row)) for row in rows]

# len считает прочитанные строки, а print выводит их количество.
print("Получено строк:", len(records))
# DataFrame создаёт таблицу; columns сохраняет заголовки даже при пустом ответе.
result_df = pd.DataFrame(records, columns=columns)
# head показывает первые строки, display красиво выводит таблицу в ноутбуке.
display(result_df.head())


## Вопросы для самопроверки

1. Почему сумму `orders.total_amount` нельзя считать после прямого соединения с `payments`?
2. Почему заказ 119 есть в первой витрине, но не может стать первой подтверждённой покупкой?
3. Почему условие `purchase_number = 1` находится в `on`, а не в итоговом `where`?